In [2]:
import os
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
con = duckdb.connect()

def load_queries(path):
    text = Path(path).read_text(encoding="utf-8")
    queries, current_name, current_sql = {}, None, []
    for line in text.splitlines():
        if line.strip().startswith("-- name:"):
            if current_name is not None:
                queries[current_name] = "\n".join(current_sql).strip()
            current_name = line.split("-- name:", 1)[1].strip()
            current_sql = []
        else:
            current_sql.append(line)
    if current_name is not None:
        queries[current_name] = "\n".join(current_sql).strip()
    return queries

def run(queries, name):
    return con.execute(queries[name]).df()

In [4]:
for f in ["sql/00_setup.sql", "sql/01_scope.sql"]:
    for name, sql in load_queries(f).items():
        if name.startswith("create_"):
            con.execute(sql)

display(run(load_queries("sql/01_scope.sql"), "scope_reconcile"))

,raw_rows,scoped_rows,rows_removed_by_scope
0,20405666,20405666,0


In [5]:
clean = load_queries("sql/01_clean.sql")
con.execute(clean["create_trips_clean"])
display(con.execute("SELECT COUNT(*) AS n_rows FROM trips_clean").df())   # must be 20,405,666

,n_rows
0,20405666


In [6]:
rc = run(clean, "rules_counts").set_index("hvfhs_license_num")
n = rc["n_rows"]
tbl = rc.drop(columns="n_rows").T
tbl["all"] = tbl.sum(axis=1)
tbl["HV0003_pct"] = (tbl["HV0003"] / n["HV0003"] * 100).round(4)
tbl["HV0005_pct"] = (tbl["HV0005"] / n["HV0005"] * 100).round(4)
tbl["all_pct"] = (tbl["all"] / n.sum() * 100).round(4)
display(tbl)

hvfhs_license_num,HV0003,HV0005,all,HV0003_pct,HV0005_pct,all_pct
wait_negative,120967,78821,199788,0.7877,1.5611,0.9791
pickup_outside_nyc,529,245,774,0.0034,0.0049,0.0038
zero_miles,2398,208,2606,0.0156,0.0041,0.0128
zero_time,4,0,4,0.0000,0.0000,0.0000
fare_negative,2393,34,2427,0.0156,0.0007,0.0119
fare_zero,832,13945,14777,0.0054,0.2762,0.0724
pay_negative,313,0,313,0.0020,0.0000,0.0015
pay_zero,4225,0,4225,0.0275,0.0000,0.0207
any_money_flag,9681,14187,23868,0.0630,0.2810,0.1170
speed_ge_100mph,1,1,2,0.0000,0.0000,0.0000


In [7]:
for name in ["rules_waterfall", "rules_overlap"]:
    print(f"\n=== {name} ===")
    display(run(clean, name))

fam = run(clean, "analysis_family_rows")
fam["pct_of_all"] = (fam["n_rows"] / 20405666 * 100).round(4)
display(fam)


=== rules_waterfall ===


,first_rule,n_rows,pct_of_all,n_uber,n_lyft
0,0_no_flag,20181669,98.9023,15225665,4956004
1,1_wait_negative (D11),199788,0.9791,120967,78821
2,2_pickup_outside_nyc,760,0.0037,517,243
3,3_zero_miles_or_time,2347,0.0115,2156,191
4,4_fare_zero_or_negative,17157,0.0841,3205,13952
5,5_pay_zero_or_negative,3945,0.0193,3945,0



=== rules_overlap ===


,wait_negative,outside_nyc,flag_zero_miles,flag_zero_time,flag_fare_negative,flag_fare_zero,flag_pay_negative,flag_pay_zero,n_rows
0,True,False,False,False,False,False,False,False,199359
1,False,False,False,False,False,True,False,False,14749
2,False,False,False,False,False,False,False,True,3640
3,False,False,False,False,True,False,False,False,2408
4,False,False,True,False,False,False,False,False,1912
5,False,True,False,False,False,False,False,False,756
6,False,False,True,False,False,False,False,True,432
7,False,False,False,False,False,False,True,False,305
8,True,False,True,False,False,False,False,False,210
9,True,False,False,False,False,False,False,True,103


,family,n_rows,pct_of_all
0,F1 volume: all rows,20405666,100.0000
1,F2 wait statistics: is_wait_valid (D11),20205878,99.0209
2,"F3 zone/borough wait tables: valid wait, picku...",20205118,99.0172
3,F4 pay level: all rows (minimum),20405666,100.0000
4,F4 pay level: no money flag (strict),20381798,99.8830
5,F5 pay per mile: miles > 0 (minimum),20403060,99.9872
6,F5 pay per mile: no money flag (strict),20381798,99.8830
7,F6 pay per minute: no money flag (strict),20381798,99.8830
8,F6 pay per minute: time > 0 (minimum),20405662,100.0000
9,F7 wait x pay joint: valid wait (minimum),20205878,99.0209


In [8]:
import pandas as pd
sens = load_queries("sql/01_clean_sensitivity.sql")

display(run(sens, "outside_nyc_pickup_effect"))

ms = run(sens, "money_sensitivity").set_index("hvfhs_license_num")
rows = []
for comp in ms.index:
    for metric in ["pay", "ppm", "ppmin"]:
        a = [float(x) for x in ms.loc[comp, f"{metric}_q_all"]]
        s = [float(x) for x in ms.loc[comp, f"{metric}_q_strict"]]
        for q, av, sv in zip(["p10", "p50", "p90"], a, s):
            rows.append((comp, metric, q, av, sv, round(100 * (sv - av) / av, 3) if av else None))
display(ms[["n_rows", "n_flagged"]])
display(pd.DataFrame(rows, columns=["company", "metric", "quantile", "all_rows", "strict", "pct_change"]))

,hvfhs_license_num,n_outside,n_outside_valid_wait,median_all_valid,median_nyc_only,p90_all_valid,p90_nyc_only,median_outside_only
0,HV0003,529,517,3.800,3.800,8.033,8.033,6.650
1,HV0005,245,243,4.083,4.083,8.033,8.033,6.033


,n_rows,n_flagged
hvfhs_license_num,,
HV0003,15356455,9681
HV0005,5049211,14187


,company,metric,quantile,all_rows,strict,pct_change
0,HV0003,pay,p10,5.670,5.670,0.000
1,HV0003,pay,p50,14.160,14.170,0.071
2,HV0003,pay,p90,37.660,37.670,0.027
3,HV0003,ppm,p10,2.817,2.818,0.035
4,HV0003,ppm,p50,4.596,4.597,0.022
5,HV0003,ppm,p90,7.836,7.836,0.000
6,HV0003,ppmin,p10,0.758,0.759,0.132
7,HV0003,ppmin,p50,0.929,0.929,0.000
8,HV0003,ppmin,p90,1.385,1.385,0.000
9,HV0005,pay,p10,5.470,5.470,0.000


In [11]:
clean = load_queries("sql/01_clean.sql")
con.execute(clean["create_trips_clean"])
display(run(clean, "final_reconciliation").T)
display(run(clean, "final_counts_by_company"))

,0
total_rows,20405666
wait_stats_rows,20205878
geo_wait_rows,20205118
pay_per_mile_rows,20403060
pay_per_minute_rows,20405662
pay_per_mile_non_null,20403060
pay_per_minute_non_null,20405662
ppm_guard_mismatch,0
ppmin_guard_mismatch,0
joint_wait_ppm_rows,20203530


,hvfhs_license_num,volume_rows,wait_stats_rows,geo_wait_rows,pay_per_mile_rows,pay_per_minute_rows,joint_wait_ppm_rows,joint_wait_ppmin_rows
0,HV0003,15356455,15235488,15234971,15354057,15356451,15233331,15235485
1,HV0005,5049211,4970390,4970147,5049003,5049211,4970199,4970390
